In [1]:
# 潜变量自回归模型
# 使用潜变量h_t总结过去信息
# p(h_t|h_t-1, x_t-1)   h  -  h_t-1  -  h_t
#                       |  \    |    \   |
# p(x_t|h_t, x_t-1)     x  -  x_t-1  -  x_t
# 也就是说每个x_t都和x_t-1和h_t相关

# 循环神经网络
# 拥有观察x, 隐变量h, 输出o
# 这里隐变量hidden和latten的区别在于hidden是没观察到存在的, latent是不存在的
# h_t 是由 x_t-1 和 h_t-1 组成的, h_t生成了o_t
# 假如观察是'你', 更新隐变量, 输出'好'; 下一个观察是'好', 更新隐变量, 输出是','
# 更新隐藏状态: h_t = Φ(W_hh*h_t-1  + W_hx * x_t-1 + b_h)
# 这里的W_hh是hidden的h_t的weight, W_hx是hidden的x_t的weight
# 输出: o_t = W_ho*h_t + b_o

# 困惑度(perplexity)
# 衡量一个语言模型的好坏可以用平均交叉熵
# π = 1/n Σ^n(-log p(x_t | x_t-1,...,x_1))
# 这里 p(x_t | x_t-1,...,x_1)表示给定前t-1个词, 第t个词是x_t的概率, 也就是基于已知词来预测下一个词
# -log就是对数概率的负数
# 1/n Σ^n 就是对整个序列所有单词的负对数概率求平均
# 历史原因NLP使用困惑度exp(π)来衡量, 是平均每次可能项, 1表示完美, 无穷大是最差情况

# 梯度裁剪
# 迭代中计算T个时间步上的梯度, 在反向传播过程中产生长度为O(T)的矩阵乘法链, 导致数值不稳定
# 梯度裁剪可以有效防止梯度爆炸
#   - 如果梯度长度超过θ, 那么拖影回长度θ
#   - g <- min(1, θ/||G||) g
#   - 如果θ/||G|| > 1, 就会把它拉回到g; 如果θ/||G|| < 1, θ/||G|| * g = θ

# 总结
# 循环神经网络的输出取决于前一时间输入和当下时间的隐变量
# 应用到语言模型中时, 循环神经网络根据当前词预测下一次时刻的输出
# 通常使用困惑度来衡量语言模型的好坏

In [2]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

batch_size, num_steps = 32, 35 # num_steps就是时间维度, T
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps) #vocab是把index转成对应的词

# 独热编码
F.one_hot(torch.tensor([0, 2]), len(vocab))
# 这里vocab的总长是28, 26个字母+
# 这里将第1和第3个字母都变成1
# 这样给一个下标, 就可以变成一个向量, 帮我添加一个维度

tensor([[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0],
        [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0]])

In [3]:
# 小批量数据, 形状是(批量大小, 时间步数)
X = torch.arange(10).reshape((2, 5))
F.one_hot(X.T, 28).shape
# 这里transpose的原因是(2,5)的one_hot不符合我们的数据处理; (5,2)可以确保在每一个时间步上, 所有的批次都能被一次性处理, 应用在
# 对时间依赖性较强的模型中

torch.Size([5, 2, 28])

In [5]:
# 初始化循环神经网络的模型参数
def get_params(vocab_size, num_hiddens, device):
    # 输入是一个个词, 通过one_hot变成向量之后, 这个向量就是一个长为vocab的向量, 所以输入的维度就是vocab_size
    # 其实就是分类, 所以类别的个数就是vocab_size
    num_inputs = num_outputs = vocab_size
    
    def normal(shape):
        return torch.randn(size=shape, device=device) * 0.01 # 把方差变成0.01
    
    W_xh = normal((num_inputs, num_hiddens))
    W_hh = normal((num_hiddens, num_hiddens))
    b_h = torch.zeros(num_hiddens, device=device)
    W_hq = normal((num_hiddens, num_outputs)) #输出
    b_q = torch.zeros(num_outputs, device=device)
    params = [W_xh, W_hh, b_h, W_hq, b_q]
    for param in params:
        param.requires_grad_(True)
    return params

In [6]:
# 一个init_rnn_state函数在初始化时返回隐藏状态
# 因为在h_0的时候, 没有上一层的隐藏状态
def init_rnn_state(batch_size, num_hiddens, device):
    return (torch.zeros((batch_size, num_hiddens), device=device),)

In [7]:
# 下面rnn函数定义了如何在一个时间步内计算隐藏状态和输出
def rnn(inputs, state, params):
    W_xh, W_hh, b_h, W_hq, b_q = params
    H, = state # 传入上一个初始化层
    outputs = []
    for X in inputs:
        H = torch.tanh(torch.mm(X, W_xh) 
                       + torch.mm(H, W_hh)
                       + b_h)
        Y = torch.mm(H, W_hq) + b_q
        outputs.append(Y)
    return torch.cat(outputs, dim=0), (H, ) # 拼出来2维的矩阵, 列数还是vocab_size, 行数变成了batch_size * time的长度
    # 还要输出更新后的隐藏状态
    # 这里其实把他拆分为一个对单一词的预测

In [8]:
# 创造一个类来包装这些函数
class RNNModelScratch:
    # 循环神经网络模型
    def __init__(self, vocab_size, num_hiddens, device, get_params, 
                 init_state, forward_fn):
        self.vocab_size = vocab_size
        self.num_hiddens = num_hiddens
        self.params = get_params(vocab_size, num_hiddens, device)
        self.init_state = init_state
        self.forward_fn = forward_fn
        
    def __call__(self, X, state):
        X = F.one_hot(X.T, self.vocab_size).type(torch.float32)
        return self.forward_fn(X, state, self.params)
    
    def begin_state(self, batch_size, device):
        return init_rnn_state(batch_size, self.num_hiddens, device)

In [11]:
# 检查输出是否是正常的形状
num_hiddens = 512
net = RNNModelScratch(len(vocab), num_hiddens, d2l.try_gpu(), get_params,
                      init_rnn_state, rnn)
state = net.begin_state(X.shape[0], d2l.try_gpu())
Y, new_state = net(X.to(d2l.try_gpu()), state)
Y.shape, len(new_state), new_state[0].shape
# 这里 2*5 = 10, 第二个维度是对每一个词下一个词的预测向量, 也就是长为28的向量
# 这里2是batch_size, 512是隐藏元更新后的隐藏状态

(torch.Size([10, 28]), 1, torch.Size([2, 512]))

In [13]:
# 定义预测函数来生成prefix后的新字符
def predict_ch8(prefix, num_preds, net, vocab, device):
    state = net.begin_state(batch_size=1, device=device)
    # vocab可以预测值map到真实的字符串词
    
    # 这里就是把第一个已知的词通过vocab转成下标, 存在outputs
    outputs = [vocab[prefix[0]]] 
    get_input = lambda: torch.tensor([outputs[-1]], device=device).reshape(
        (1, 1)) # 这里是把outputs的最后一个词作为下一个词的输入
    
    # 预热期
    for y in prefix[1:]:
        _, state = net(get_input(), state) # 因为开始的时候不需要预测, 这里是把prefix的信息放到state里面
        # 这里的_,表示我们不care输出
        outputs.append(vocab[y])
    
    for _ in range(num_preds):
        y, state = net(get_input(), state) 
        outputs.append(int(y.argmax(dim=1).reshape(1))) # 把最大的拿出来, 直接放到output里面
    
    return ''.join(vocab.idx_to_token[i] for i in outputs) # 最后一次性转成词
        
predict_ch8('time traveller ', 10, net, vocab, d2l.try_gpu())

'time traveller nlakwphhhh'

In [15]:
# 梯度剪裁
def grad_clipping(net, theta):
    if isinstance(net, nn.Module):
        params = [p for p in net.parameters() if requires_grad]
    else:
        params = net.params # 所有层都拿了出来
    norm = torch.sqrt(sum(torch.sum(
            (p.grad**2)) for p in params)) # 把所有的层里面的梯度**2, 求和, 然后对所有的层求和, 再开根号
    # 等价于是把所有层的所有参数拿出来, 做了个norm
    if norm > theta: # 如果大于阈值
        for param in params:
            param.grad[:] *= theta / norm
            # 把参数里面所有梯度都拉到theta一样大

In [17]:
# 定义一个函数在一个迭代周期内训练模型
def train_epoch_ch8(net, train_iter, loss, updater, device, use_random_iter):
    # use_random_iter在文本里是没有任何关系的
    state, timer = None, d2l.Timer()
    metric = d2l.Accumulator(2)
    
    for X, Y in train_iter:
        if state is None or use_random_iter: # 因为use_random_iter里的sequence互相不连续, 所以当前的批量从头开始
            state = net.begin_state(batch_size=X.shape[0], device=device) # X[0]是批量大小
        else:  # else 不做初始化
            # 因为sequence_iter说明是连续的
            if isinstance(net. nn.Module) and not isinstance(state, tuple):
                state.detach_() # detach意思是做梯度的时候只做现在和之后的那些
            else:
                for s in state:
                    s.detach()
            # 这里只保留数值, 但是不做梯度更新
        y = Y.T.reshape(-1) # 把时间序列放到前面, 拉成一个向量
        X, y = X.to(device), y.to(device)
        y_hat, state = net(X, state) 
        l = loss(y_hat, y.long()).mean() # 因为在net里我们已经处理过, 把y_hat拉成了一个1维向量
        
        if isinstance(updater, torch.optim.Optimizer):
            updater.zero_grad()
            l.backward()
            grad_clipping(net, 1)
            updater.step()
        else:
            l.backward()
            grad_clipping(net, 1)
            updater(batch_size=1)
        metric.add(l * y.numel(), y.numel())
    return math.exp(metric[0] / metric[1]), metric[1] / timer.stop() # Loss的累加/样本就得到平均的Loss

In [18]:
# 循环神经网络模型的训练函数既支持从零开始实现, 也可以使用高级API来实现
def train_ch8(net, train_iter, vocab, lr, num_epoch, device, use_random_iter=False):
    loss = nn.CrossEntropyLoss()
    animator = animator = d2l.Animator(xlabel='epoch', ylabel='perplexity',
                            legend=['train'], xlim=[10, num_epochs])
    if isinstance(net, nn.Module):
        updater = torch.optim.SGD(net.parameters(), lr)
    else:
        updater = lambda batch_size: d2l.sgd(net.params, lr, batch_size)
    predict = lambda prefix: predict_ch8(prefix, 50, net, vocab, device)
    for epoch in range(num_epoch):
        ppl, speed = train_epoch_ch8(net, train_iter, loss, updater, device, use_random_iter)
        if (epoch + 1) % 10 == 0:
            print(predict('time traveller'))
            animator.add(epoch + 1, [ppl])
    print(f'困惑度 {ppl:.1f}, {speed:.1f} 标记/秒 {str(device)}')
    print(predict('time traveller'))
    print(predict('traveller'))

In [ ]:
num_epochs, lr = 500, 1
train_ch8(net, train_iter, vocab, lr, num_epochs, d2l.try_gpu())

In [ ]:
train_ch8(net, train_iter, vocab, lr, num_epochs, d2l.try_gpu(),
          use_random_iter=True)

In [19]:
# 简洁实现
import torch
from torch import nn
from torch.nn import functional as F
from d2l import torch as d2l

batch_size, num_steps = 32, 35
train_iter, vocab = d2l.load_data_time_machine(batch_size, num_steps)



In [20]:
num_hiddens = 256
rnn_layer = nn.RNN(len(vocab), num_hiddens)

In [21]:
# 使用张量来初始化隐藏状态
state = torch.zeros((1, batch_size, num_hiddens))
state.shape #这里的1没啥意义

torch.Size([1, 32, 256])

In [22]:
# 通过一个隐藏状态和一个输入, 我们可以用更新后的隐藏状态计算输出
X = torch.rand(size=(num_steps, batch_size, len(vocab)))
Y, state_new = rnn_layer(X, state)
Y.shape, state_new.shape
# 这里的Y的第一个维度是时间, 第二个维度是批量, 第三个维度是输出

(torch.Size([35, 32, 256]), torch.Size([1, 32, 256]))

In [28]:
# 为一个完整的循环神经网络定义了一个RNNModel类
class RNNModel(nn.Module):
    def __init__(self, rnn_layer, vocab_size, **kwargs):
        super(RNNModel, self).__init__(**kwargs)
        self.rnn = rnn_layer
        self.vocab_size = vocab_size
        self.num_hiddens = self.rnn.hidden_size
        # 这里不同的是没包括输出层
        if not self.rnn.bidirectional:
            self.num_directions = 1
            self.linear = nn.Linear(self.num_hiddens, self.vocab_size)
        else:
            self.num_directions = 2
            self.linear = nn.Linear(self.num_hiddens, self.vocab_size)
            
    def forward(self, inputs, state):
        X = F.one_hot(inputs.T.long(), self.vocab_size)
        X = X.to(torch.float32)
        Y, state = self.rnn(X, state) # Y 是时间步数*批量大小*隐藏大小
        output = self.linear(Y.reshape((-1, Y.shape[-1])))
        return output, state
    
    def begin_state(self, device, batch_size=1):
        if not isinstance(self.rnn, nn.LSTM):
            return torch.zeros((self.num_directions * self.rnn.num_layers,
                                batch_size, self.num_hiddens), device=device)
        else:
            return (torch.zeros((self.num_directions * self.rnn.num_layers,
                                 batch_size, self.num_hiddens),
                                device=device),
                    torch.zeros((
                        self.num_directions * self.rnn.num_layers,
                        batch_size, self.num_hiddens), device=device))

In [29]:
# 基于一个具有随机权重的模型进行预测
device = d2l.try_gpu()
net = RNNModel(rnn_layer, vocab_size=len(vocab))
net = net.to(device)
d2l.predict_ch8('time traveller', 10, net, vocab, device)

'time travellervwbbwvzbbw'

In [ ]:
num_epochs, lr = 500, 1
d2l.train_ch8(net, train_iter, vocab, lr, num_epochs, device)